# Interpolant vs DiffusionSR on ORCHARD: the 13-arm experiment

Results of the `slurm/interpolant/` experiment as run on ORCHARD (October 2026): 13 trained arms, all
evaluated on the 3,829-frame test split with K = 10 samples and a matched 20-step sampling budget,
plus the frozen encoder on its own as a reference.

| group | tasks | what varies |
|---|---|---|
| DDPM | 0, 11 | 0 = DDPM in the shared harness (the attributable baseline); 11 = DiffusionSR as published |
| SI enc | 1-5 | stochastic interpolant from the bicubic start, conditioned on the frozen encoder (Option A sigma sweep, Option B c sweep) |
| SI enc-start | 12 | task 2 with the transport starting at the encoder's own HR prediction (`si_base: enc`) |
| SI ulr | 6-10 | paper-style twins of 1-5: no encoder, upscaled-LR conditioning |
| reference | - | the frozen encoder alone (no generative model) |

Separate from `19_interpolant_vs_diffusionsr.ipynb`, which predates these runs and only covers tasks 0-5 and 11.
Read section 7 before drawing conclusions. **Section 8 is a focused head-to-head of tasks 0, 2 and 12.**

In [ ]:
# -- Configuration ----------------------------------------------------------------
import os, sys
from pathlib import Path

# ORCHARD paths. The repo and data live on /project; ~/LPBFDiffusionSR_dev is a symlink to REPO.
REPO      = Path('/project/community/ngng/LPBFDiffusionSR_dev')
RUNS_ROOT = REPO / 'diffusionsr' / 'runs' / 'interpolant'
EVAL_ROOT = REPO / 'eval_results' / 'interpolant'
CACHE     = REPO / 'eval_results' / '_predictions_cache'   # per-arm pred.npy / gt.npy, memory-mapped
WANDB_ENTITY  = os.getenv('WANDB_ENTITY') or 'ngng-'
WANDB_PROJECT = 'Flow3D_SuperResolution'

# Repo-standard FIXED temperature scale (same as notebook 18 and train_rrdn_encoder.py), so these figures
# are comparable with every other temperature figure in the repo. Do not switch to a data-derived range.
VMIN_T, VMAX_T = 293, 5000
MP_THRESHOLD_K = 1900.0   # melt-pool boundary, as in analysis/evaluate_coupled.py
CH = 0                    # temperature channel

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

In [ ]:
# -- Imports -----------------------------------------------------------------------
import json, tempfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
try:
    import ipywidgets as widgets
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
# NB21_STATIC=1 renders one fixed frame instead of sliders, for a pre-executed copy (sliders need a live kernel).
if os.getenv('NB21_STATIC'):
    HAVE_WIDGETS = False
warnings.filterwarnings('ignore', category=RuntimeWarning)

In [ ]:
# -- Arms --------------------------------------------------------------------------
# (task, label, directory name, group). The group drives colours and which comparisons are attributable.
ARMS = [
    ('ref', 'Encoder alone',                  'encoder_only',                      'reference'),
    (0,     'DDPM (matched harness)',         'si_baseline_ddpm_matched_temp',     'DDPM'),
    (11,    'DiffusionSR (as published)',     'si_baseline_diffusionsr_temp',      'DDPM'),
    (1,     'SI-A s=0.05 enc',                'si_si_opt_a_sigma005_enc_temp',     'SI enc'),
    (2,     'SI-A s=0.10 enc',                'si_si_opt_a_sigma010_enc_temp',     'SI enc'),
    (3,     'SI-A s=0.20 enc',                'si_si_opt_a_sigma020_enc_temp',     'SI enc'),
    (4,     'SI-B c=0.10 enc',                'si_si_opt_b_c010_enc_temp',         'SI enc'),
    (5,     'SI-B c=0.25 enc',                'si_si_opt_b_c025_enc_temp',         'SI enc'),
    (12,    'SI-A s=0.10 enc, encoder start', 'si_si_opt_a_sigma010_encbase_temp', 'SI enc-start'),
    (6,     'SI-A s=0.05 ulr',                'si_si_opt_a_sigma005_ulr_temp',     'SI ulr'),
    (7,     'SI-A s=0.10 ulr',                'si_si_opt_a_sigma010_ulr_temp',     'SI ulr'),
    (8,     'SI-A s=0.20 ulr',                'si_si_opt_a_sigma020_ulr_temp',     'SI ulr'),
    (9,     'SI-B c=0.10 ulr',                'si_si_opt_b_c010_ulr_temp',         'SI ulr'),
    (10,    'SI-B c=0.25 ulr',                'si_si_opt_b_c025_ulr_temp',         'SI ulr'),
]
GROUP_COLOR = {'reference': '#555555', 'DDPM': '#cc3333', 'SI enc': '#3366cc',
               'SI enc-start': '#00aa77', 'SI ulr': '#cc8800'}

arms = {}
for task, label, run, group in ARMS:
    res = EVAL_ROOT / run / f'{run}_results.json'
    if res.exists():
        arms[label] = dict(task=task, run=run, group=group, results=res,
                           eval_dir=EVAL_ROOT / run, run_dir=RUNS_ROOT / run)
    else:
        print(f'not evaluated yet, skipped: [{task}] {label}')
ORDER = list(arms)
print(f'{len(arms)} of {len(ARMS)} arms loaded')

## 1 - Metrics

Every number comes from the same code, `compute_metrics()` in `analysis/evaluate_coupled.py`, on the
same test frames, in physical units:

- **MAE / RMSE** - per sample, averaged over the K samples and N frames (K). **Ensemble MAE** scores the mean of the K samples.
- **MP / KH MAE** - error in melt-pool depth and keyhole-depression depth (um), from the 1900 K region.
- **area MAE** - error in melt-pool area (um^2).
- **calibration_2sigma** - fraction of frames with a melt pool whose true depth lies within the ensemble's mean +- 2 std. Ideal is about **0.95**, not 1.

Bold = best per column (closest to 0.95 for calibration; the encoder row has no ensemble, so it is excluded there).
`depth_mae_um` is identical to `MP_MAE_mean` by construction and is omitted.

In [ ]:
# -- Metrics table -----------------------------------------------------------------
SHOW = ['MAE_mean', 'MAE_std', 'MAE_ensemble_mean', 'RMSE_mean',
        'MP_MAE_mean', 'KH_MAE_mean', 'area_mae_um2', 'calibration_2sigma']
rows = {}
for label in ORDER:
    r = json.load(open(arms[label]['results']))
    rows[label] = {'task': arms[label]['task'], 'group': arms[label]['group'],
                   **{k: r.get(k, np.nan) for k in SHOW}}
    if arms[label]['group'] == 'reference':
        rows[label]['calibration_2sigma'] = np.nan   # K = 1: no ensemble to be calibrated
metrics = pd.DataFrame(rows).T
for k in SHOW:
    metrics[k] = pd.to_numeric(metrics[k], errors='coerce')

def _best(col):
    s = metrics[col].dropna()
    return (s - 0.95).abs().idxmin() if col == 'calibration_2sigma' else s.idxmin()
BEST = {c: _best(c) for c in SHOW if c != 'MAE_std'}

def _highlight(col):
    return ['font-weight: bold; color: #0a7' if BEST.get(col.name) == idx else '' for idx in col.index]

display(metrics.style.format({k: '{:.4g}' for k in SHOW}).apply(_highlight, axis=0))

In [ ]:
# -- The questions the design can answer, one change at a time ---------------------
PAIRS = [
    ('Generative process: DDPM -> interpolant (tasks 0 -> 2)', 'DDPM (matched harness)', 'SI-A s=0.10 enc'),
    ('Start point: bicubic -> encoder prediction (tasks 2 -> 12)', 'SI-A s=0.10 enc', 'SI-A s=0.10 enc, encoder start'),
    ('Best interpolant vs DDPM baseline (tasks 0 -> 12)', 'DDPM (matched harness)', 'SI-A s=0.10 enc, encoder start'),
    ('Training harness alone, both DDPM (tasks 11 -> 0)', 'DiffusionSR (as published)', 'DDPM (matched harness)'),
    ('Adding a generative stage at all (encoder -> task 0)', 'Encoder alone', 'DDPM (matched harness)'),
]
CMP_METRICS = ['MAE_ensemble_mean', 'MP_MAE_mean', 'KH_MAE_mean', 'area_mae_um2']
out = []
for question, a, b in PAIRS:
    if a not in metrics.index or b not in metrics.index:
        continue
    for m in CMP_METRICS:
        va, vb = metrics.loc[a, m], metrics.loc[b, m]
        out.append({'question': question, 'metric': m, 'from': va, 'to': vb,
                    'change': 100.0 * (vb - va) / va if va else np.nan,
                    'verdict': 'better' if vb < va else 'worse'})
cmp = pd.DataFrame(out)
display(cmp.style.format({'from': '{:.4g}', 'to': '{:.4g}', 'change': '{:+.1f}%'}).hide(axis='index'))

In [ ]:
# -- Key metrics per arm, coloured by group ----------------------------------------
PLOT_METRICS = [('MAE_ensemble_mean', 'ensemble MAE (K)'), ('MP_MAE_mean', 'melt-pool depth MAE (um)'),
                ('KH_MAE_mean', 'keyhole depth MAE (um)'), ('area_mae_um2', 'melt-pool area MAE (um^2)')]
fig, axes = plt.subplots(1, len(PLOT_METRICS), figsize=(19, 0.38 * len(ORDER) + 1.2))
for i, (ax, (m, title)) in enumerate(zip(axes, PLOT_METRICS)):
    v = metrics[m]
    ax.barh(range(len(v)), v.values, color=[GROUP_COLOR[metrics.loc[l, 'group']] for l in v.index])
    ax.set_yticks(range(len(v)))
    ax.set_yticklabels([f"[{metrics.loc[l, 'task']}] {l}" for l in v.index] if i == 0 else [], fontsize=8)
    ax.invert_yaxis()
    ax.set_title(title, fontsize=10)
    ax.grid(axis='x', alpha=0.3)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in GROUP_COLOR.values()]
fig.legend(handles, list(GROUP_COLOR), loc='lower center', ncol=len(GROUP_COLOR), fontsize=8,
           bbox_to_anchor=(0.5, -0.04))
plt.tight_layout()
plt.show()

## 2 - Training

Loss **values are not comparable across processes**: the DDPM arms regress onto the noise epsilon, the
interpolant arms onto the velocity x1 - x0, and the encoder-start arm's velocity is smaller still because it
starts closer to the target. Compare curves within a panel only. Dots mark the best (lowest validation) epoch,
which is the checkpoint each eval used.

(Training wall-clock is not shown: `training_time.txt` restarts its timer on every requeue, so for these
heavily preempted runs it records only the last segment.)

In [ ]:
# -- Loss curves, per group --------------------------------------------------------
GROUPS = ['DDPM', 'SI enc', 'SI enc-start', 'SI ulr']
fig, axes = plt.subplots(2, len(GROUPS), figsize=(19, 7.5))
summary = {}
for j, g in enumerate(GROUPS):
    for label in ORDER:
        a = arms[label]
        if a['group'] != g:
            continue
        for i, fname in enumerate(['loss_epoch.txt', 'validation_loss_epoch.txt']):
            f = a['run_dir'] / fname
            if not f.exists():
                continue
            y = np.loadtxt(f).reshape(-1)
            line, = axes[i, j].plot(y, lw=1, label=f"[{a['task']}] {label}")
            if i == 1:
                b = int(np.argmin(y))
                axes[i, j].plot(b, y[b], 'o', ms=4, color=line.get_color())
                summary[label] = {'task': a['task'], 'epochs': len(y), 'best epoch': b,
                                  'best val loss': y[b], 'final val loss': y[-1],
                                  'final / best': y[-1] / y[b]}
    for i, kind in enumerate(['train', 'validation']):
        ax = axes[i, j]
        ax.set_yscale('log')
        ax.set_title(f'{g}: {kind} loss', fontsize=10)
        ax.set_xlabel('epoch')
        if ax.lines:
            ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

print('Validation summary. final / best >> 1 means the run got worse after its best epoch (overfitting).')
display(pd.DataFrame(summary).T.style.format({'best val loss': '{:.4g}', 'final val loss': '{:.4g}',
                                              'final / best': '{:.2f}'}))

## 3 - Prediction arrays

Each eval uploaded `predictions.npz` (`pred` as (N, K, C, H, W), `gt` as (N, C, H, W), physical units) to its
W&B run as `<run>_eval_predictions`. The first run of this cell downloads each one and stores `pred.npy` /
`gt.npy` under `eval_results/_predictions_cache/`; later runs memory-map them, so nothing close to the
~7 GB total is held in RAM. The encoder-alone arm's file is local (`evaluate_encoder_only.py`).

In [ ]:
# -- Load (and on first use, fetch and cache) every arm's predictions --------------
def cached_arrays(run, eval_dir):
    d = CACHE / run
    if not ((d / 'pred.npy').exists() and (d / 'gt.npy').exists()):
        d.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(dir=CACHE, ignore_cleanup_errors=True) as tmp:
            src = eval_dir / 'predictions.npz'
            if not src.exists():
                import wandb
                art = wandb.Api().artifact(f'{WANDB_ENTITY}/{WANDB_PROJECT}/{run}_eval_predictions:latest')
                art.download(root=tmp)
                src = Path(tmp) / 'predictions.npz'
            with np.load(src, allow_pickle=True) as z:   # `with`: an open file blocks temp cleanup on NFS
                np.save(d / 'pred.npy', z['pred'])
                np.save(d / 'gt.npy', z['gt'])
    return np.load(d / 'pred.npy', mmap_mode='r'), np.load(d / 'gt.npy', mmap_mode='r')

CACHE.mkdir(parents=True, exist_ok=True)
pred, gt = {}, None
for label in ORDER:
    try:
        p, g = cached_arrays(arms[label]['run'], arms[label]['eval_dir'])
    except Exception as e:
        print(f'no predictions for {label}: {str(e).splitlines()[0][:120]}')
        continue
    pred[label] = p
    if gt is None:
        gt = g
    elif g.shape != gt.shape or not np.array_equal(g[::500], gt[::500]):
        raise RuntimeError(f'{label}: ground truth differs from the other arms -- not the same test frames')

N_FRAMES = 0 if gt is None else gt.shape[0]
HOT = (np.asarray(gt[:, CH]) > MP_THRESHOLD_K).sum(axis=(1, 2)) if N_FRAMES else np.array([])
DEFAULT_FRAME = int(np.argmax(HOT)) if N_FRAMES else 0   # the frame with the largest melt pool
print(f'{len(pred)} arm(s), {N_FRAMES} test frames, {int((HOT > 0).sum())} with a melt pool; '
      f'default frame {DEFAULT_FRAME} ({int(HOT[DEFAULT_FRAME]) if N_FRAMES else 0} melted pixels)')

def with_frame_slider(render_fn, default=DEFAULT_FRAME):
    # Redraw on slider release; without ipywidgets, render one static frame.
    if N_FRAMES == 0:
        return
    if not HAVE_WIDGETS:
        render_fn(default)
        return
    slider = widgets.IntSlider(min=0, max=N_FRAMES - 1, value=default, description='frame:',
                               continuous_update=False, layout=widgets.Layout(width='460px'))
    display(widgets.VBox([slider, widgets.interactive_output(render_fn, {'frame': slider})]))

## 4 - Overlays

All panels share one frame and the repo's fixed temperature scale (293-5000 K), so they can be compared by eye.
Columns: ground truth, three individual samples, the ensemble mean, its absolute error, and the per-pixel
ensemble standard deviation (error and std each on one scale shared by every row, clipped at the 99th
percentile so a single spatter pixel does not wash out the rest).
Edit `SHOW_ARMS` to change the rows.

In [ ]:
# -- Rebuilt overlay: shared frame, shared scales ----------------------------------
SHOW_ARMS = ['Encoder alone', 'DDPM (matched harness)', 'DiffusionSR (as published)', 'SI-A s=0.10 enc',
             'SI-B c=0.25 enc', 'SI-A s=0.10 enc, encoder start', 'SI-A s=0.10 ulr']
N_SAMPLES_SHOWN = 3

def render_overlay(frame):
    labels = [l for l in SHOW_ARMS if l in pred]
    g = np.asarray(gt[frame, CH])
    P = {l: np.asarray(pred[l][frame, :, CH]) for l in labels}          # (K, H, W) each
    # Shared error/std scales clipped at the 99th percentile over all rows: a single isolated hot pixel in
    # the ground truth (spatter, ~8000 K) otherwise sets the maximum and leaves every other pixel near black.
    err_max = float(np.percentile(np.stack([np.abs(P[l].mean(0) - g) for l in labels]), 99)) or 1.0
    std_max = float(np.percentile(np.stack([P[l].std(0) for l in labels]), 99)) or 1.0
    ncol = 1 + N_SAMPLES_SHOWN + 3
    fig, axes = plt.subplots(len(labels), ncol, figsize=(2.2 * ncol, 2.2 * len(labels)), squeeze=False)
    for r, l in enumerate(labels):
        p, m = P[l], P[l].mean(0)
        panels = [(g, 'ground truth', 'jet', VMIN_T, VMAX_T)]
        panels += [(p[k], f'sample {k + 1}', 'jet', VMIN_T, VMAX_T) if k < len(p) else None
                   for k in range(N_SAMPLES_SHOWN)]
        panels += [(m, 'ensemble mean', 'jet', VMIN_T, VMAX_T),
                   (np.abs(m - g), f'|mean - GT|, 0-{err_max:.0f} K (p99)', 'magma', 0, err_max),
                   (p.std(0), f'ensemble std, 0-{std_max:.0f} K (p99)', 'hot', 0, std_max)]
        # Column titles come from a fixed list, not from row 0's panels: row 0 is the encoder (K = 1),
        # whose sample-2/3 panels are blank, which left those columns untitled.
        col_titles = (['ground truth'] + [f'sample {k + 1}' for k in range(N_SAMPLES_SHOWN)]
                      + ['ensemble mean', f'|mean - GT|, 0-{err_max:.0f} K (p99)',
                         f'ensemble std, 0-{std_max:.0f} K (p99)'])
        for c, panel in enumerate(panels):
            ax = axes[r, c]
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(col_titles[c], fontsize=8)
            if panel is None:
                ax.axis('off')
                continue
            img, title, cmap, vmin, vmax = panel
            ax.imshow(img.T, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax)
        axes[r, 0].set_ylabel(f"[{arms[l]['task']}] {l}", fontsize=7)
    fig.suptitle(f'Frame {frame} - temperature (K), fixed scale {VMIN_T}-{VMAX_T} K', fontsize=10)
    plt.tight_layout()
    plt.show()

with_frame_slider(render_overlay)

In [ ]:
# -- The overlay PNG each eval rendered itself (frame 70, per-run colour scale: do NOT compare by eye) --
for l in [x for x in SHOW_ARMS if x in arms]:
    png = arms[l]['eval_dir'] / 'overlay_frame70.png'
    if png.exists():
        print(f"[{arms[l]['task']}] {l}")
        display(Image(filename=str(png), width=1100))

## 5 - Melt-pool boundaries

The physically meaningful view: the 1900 K isotherm. Left, the boundary of each arm's ensemble mean against the
ground truth (black). Right, every individual sample's boundary for two arms, which shows how much the samples
disagree about the melt pool. The view is cropped to the melt region of the ground truth.

In [ ]:
# -- Melt-pool contours, same slider ------------------------------------------------
PALETTE = ['#00d5ff', '#cc3333', '#ff7f0e', '#3366cc', '#9467bd', '#00aa77', '#cc8800', '#17becf']
SPREAD_ARMS = ['DDPM (matched harness)', 'SI-A s=0.10 enc, encoder start']

def _crop(ax, g, margin=6):
    hot = np.argwhere(g > MP_THRESHOLD_K)        # (x, z) indices in the un-transposed field
    if len(hot):
        (x0, z0), (x1, z1) = hot.min(0), hot.max(0)
        ax.set_xlim(max(x0 - margin, 0) - 0.5, min(x1 + margin, g.shape[0] - 1) + 0.5)
        ax.set_ylim(max(z0 - margin, 0) - 0.5, min(z1 + margin, g.shape[1] - 1) + 0.5)

def render_contours(frame):
    g = np.asarray(gt[frame, CH])
    spread = [l for l in SPREAD_ARMS if l in pred]
    fig, axes = plt.subplots(1, 1 + len(spread), figsize=(6 * (1 + len(spread)), 5.5), squeeze=False)
    axes = axes[0]
    for ax in axes:
        ax.imshow(g.T, origin='lower', cmap='Greys_r', vmin=VMIN_T, vmax=VMAX_T)
        if g.max() > MP_THRESHOLD_K:
            ax.contour(g.T, levels=[MP_THRESHOLD_K], colors='k', linewidths=2.5)
        _crop(ax, g)
    handles = [plt.Line2D([], [], color='k', lw=2.5, label='ground truth')]
    for i, l in enumerate([x for x in SHOW_ARMS if x in pred]):
        m = np.asarray(pred[l][frame, :, CH]).mean(0)
        col = PALETTE[i % len(PALETTE)]
        if m.max() > MP_THRESHOLD_K:
            axes[0].contour(m.T, levels=[MP_THRESHOLD_K], colors=[col], linewidths=1.3)
        handles.append(plt.Line2D([], [], color=col, label=f"[{arms[l]['task']}] {l}"))
    axes[0].legend(handles=handles, fontsize=7, loc='lower left')
    axes[0].set_title(f'Frame {frame}: 1900 K boundary of each ensemble mean', fontsize=10)
    for ax, l in zip(axes[1:], spread):
        p = np.asarray(pred[l][frame, :, CH])
        for k in range(len(p)):
            if p[k].max() > MP_THRESHOLD_K:
                ax.contour(p[k].T, levels=[MP_THRESHOLD_K], colors=['#cc3333'], linewidths=0.8, alpha=0.7)
        ax.set_title(f"[{arms[l]['task']}] {l}: all {len(p)} samples (red)", fontsize=10)
    plt.tight_layout()
    plt.show()

with_frame_slider(render_contours)

## 6 - Ensemble spread

What DiffusionSR exists to provide: a spread of plausible fields, not one answer. A method that lowers MAE by
collapsing that spread is a regression, not a win. Maps share one scale; the table averages over the whole
test set, overall and inside the true melt pool, next to calibration (ideal ~0.95).

In [ ]:
# -- Per-pixel ensemble std, same slider ---------------------------------------------
ENS_ARMS = [l for l in SHOW_ARMS if l in pred and pred[l].shape[1] > 1]

def render_std(frame):
    S = {l: np.asarray(pred[l][frame, :, CH]).std(0) for l in ENS_ARMS}
    vmax = float(np.percentile(np.stack(list(S.values())), 99)) or 1.0   # p99: see the overlay cell
    fig, axes = plt.subplots(1, len(S), figsize=(3.1 * len(S), 3.3), squeeze=False)
    for ax, (l, s) in zip(axes[0], S.items()):
        im = ax.imshow(s.T, origin='lower', cmap='hot', vmin=0, vmax=vmax)
        ax.set_title(f"[{arms[l]['task']}] {l}", fontsize=7)
        ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=axes[0].tolist(), fraction=0.02, label='std (K)')
    fig.suptitle(f'Frame {frame}: per-pixel ensemble std, shared scale 0-{vmax:.0f} K (clipped at p99)', fontsize=10)
    plt.show()

with_frame_slider(render_std)

# Test-set spread per arm, streamed in chunks so the memory-mapped arrays are never fully loaded.
spread = {}
melt_mask_all = np.asarray(gt[:, CH]) > MP_THRESHOLD_K
for l, p in pred.items():
    if p.shape[1] < 2:
        continue
    tot = tot_mp = 0.0
    for s in range(0, N_FRAMES, 256):
        sd = np.asarray(p[s:s + 256, :, CH], dtype=np.float32).std(axis=1)
        tot += float(sd.sum())
        tot_mp += float(sd[melt_mask_all[s:s + 256]].sum())
    spread[l] = {'task': arms[l]['task'],
                 'mean std, all pixels (K)': tot / sd[0].size / N_FRAMES,
                 'mean std, melt-pool pixels (K)': tot_mp / max(int(melt_mask_all.sum()), 1),
                 'calibration_2sigma': metrics.loc[l, 'calibration_2sigma'],
                 'ensemble MAE (K)': metrics.loc[l, 'MAE_ensemble_mean']}
display(pd.DataFrame(spread).T.style.format({c: '{:.3g}' for c in
        ['mean std, all pixels (K)', 'mean std, melt-pool pixels (K)', 'calibration_2sigma', 'ensemble MAE (K)']}))

## 7 - Reading this correctly

- **Tasks 0 vs 2 is the attributable process comparison** (configs differ only in `si_process`), and **2 vs 12
  isolates the start point** (only `si_base`). Tasks 6-10 change encoder, conditioning and process at once
  relative to task 0, so compare them with each other or with their enc twins only.
- **Task 11 is a reference, not the baseline.** It is DiffusionSR's original loop (no EMA, fixed LR, no flips);
  its validation loss bottoms out around epoch 46 and then rises about 15x, so it is evaluated from that early
  checkpoint. Task 0 is the same DDPM in the shared harness.
- **The encoder-alone row is the bar to clear.** Every arm conditions on (or starts from) this encoder.
- **Best-epoch selection is noisy for the interpolant arms** (their validation loss fluctuates strongly), which
  adds some luck to which checkpoint each eval used; treat small gaps between SI arms with caution.
- **Physics metrics in um assume 10 um per HR pixel** (`PIXEL_SIZE_UM['2x']` in `evaluate_coupled.py`); this was
  not verified against the simulation setup. Rankings do not depend on it. MP/KH/area errors average over all
  frames, including the few without melt.
- **Provenance.** Tasks 0-5 and 7-10 were re-scored from their saved predictions after the melt-pool metric fix
  (`rescore_predictions.py`; MAE reproduced exactly; the pre-fix files are `*_results.before_rescore.json`).
  Task 6 was re-evaluated (its predictions upload was lost), so its samples are a fresh draw. Task 11 was
  re-evaluated with the intended 20 DDIM steps after a bug made it sample 1000. The `results.json` inside the
  W&B artifacts of tasks 0-10 is the pre-fix version; the local files read here are the corrected ones.

## 8 - Head-to-head: task 0 vs task 2 vs task 12

The three arms the design compares directly, on their own:

| task | what it is | differs from task 0 by |
|---|---|---|
| **0** | DDPM, conditioned on the encoder features (DiffusionSR in the shared harness) | - |
| **2** | interpolant from the **bicubic** start, same encoder conditioning | the generative process only (`si_process`) |
| **12** | interpolant from the **encoder's prediction**, same encoder conditioning | the process, and the start point (`si_base: enc`) |

So 0 -> 2 isolates the generative process, and 2 -> 12 isolates the start point.

Below: the metrics side by side, then for each frame **all 10 samples** of each arm with the ensemble mean, its error and
the ensemble spread, then the melt-pool boundaries of all 10 samples of each arm. Images are cropped to the melt region.
A pre-rendered copy shows three frames (the largest melt pool in the test set, a median-sized one and a small one);
run live, a slider picks any frame.

In [ ]:
# -- Head-to-head metrics: values and change relative to task 0 ----------------------
H2H = [l for l in ['DDPM (matched harness)', 'SI-A s=0.10 enc', 'SI-A s=0.10 enc, encoder start'] if l in arms]
H2H_COLOR = {'DDPM (matched harness)': '#cc3333', 'SI-A s=0.10 enc': '#3366cc',
             'SI-A s=0.10 enc, encoder start': '#00aa77'}
H2H_NAME = {l: f"task {arms[l]['task']}" for l in H2H}
H2H_METRICS = [('MAE_ensemble_mean', 'ensemble MAE (K)'), ('MAE_mean', 'per-sample MAE (K)'),
               ('RMSE_mean', 'RMSE (K)'), ('MP_MAE_mean', 'melt-pool depth MAE (um)'),
               ('KH_MAE_mean', 'keyhole depth MAE (um)'), ('area_mae_um2', 'melt-pool area MAE (um^2)'),
               ('calibration_2sigma', 'calibration (ideal ~0.95)')]
base = H2H[0]
table = {}
for l in H2H:
    col = {}
    for m, name in H2H_METRICS:
        v, v0 = metrics.loc[l, m], metrics.loc[base, m]
        col[name] = f'{v:.4g}' if l == base else f'{v:.4g}  ({100.0 * (v - v0) / v0:+.0f}%)'
    if l in spread:
        s, s0 = spread[l]['mean std, melt-pool pixels (K)'], spread[base]['mean std, melt-pool pixels (K)']
        col['sample spread in melt pool (K)'] = f'{s:.3g}' if l == base else f'{s:.3g}  ({100.0 * (s - s0) / s0:+.0f}%)'
    table[f"{H2H_NAME[l]}: {l}"] = col
print('Brackets: change relative to task 0. Lower is better for every error row; calibration is best near 0.95.')
display(pd.DataFrame(table))

# The same four headline metrics as bars, one panel each.
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, (m, name) in zip(axes, [H2H_METRICS[0], H2H_METRICS[3], H2H_METRICS[4], H2H_METRICS[5]]):
    vals = [metrics.loc[l, m] for l in H2H]
    bars = ax.bar([H2H_NAME[l] for l in H2H], vals, color=[H2H_COLOR[l] for l in H2H])
    ax.bar_label(bars, fmt='%.3g', fontsize=8)
    ax.set_title(name, fontsize=10)
    ax.set_ylim(0, max(vals) * 1.18)
plt.tight_layout()
plt.show()

In [ ]:
# -- Frames for the head-to-head: largest, median-sized and small melt pool ----------
melted = np.where(HOT > 0)[0]
by_size = melted[np.argsort(HOT[melted], kind='stable')]
H2H_FRAMES = {'largest melt pool': int(by_size[-1]), 'median melt pool': int(by_size[len(by_size) // 2]),
              'small melt pool': int(by_size[len(by_size) // 10])}
for k, f in H2H_FRAMES.items():
    print(f'{k:18s} frame {f:5d}  ({int(HOT[f])} melted pixels)')

def show_frames(render_fn):
    # Live: one slider. Pre-rendered (NB21_STATIC): every frame in H2H_FRAMES.
    if HAVE_WIDGETS:
        with_frame_slider(render_fn)
    else:
        for f in H2H_FRAMES.values():
            render_fn(f)

def _melt_box(g, margin=6):
    hot = np.argwhere(g > MP_THRESHOLD_K)        # (x, z) indices of the un-transposed field
    if not len(hot):
        return None
    (x0, z0), (x1, z1) = hot.min(0), hot.max(0)
    return (max(x0 - margin, 0) - 0.5, min(x1 + margin, g.shape[0] - 1) + 0.5,
            max(z0 - margin, 0) - 0.5, min(z1 + margin, g.shape[1] - 1) + 0.5)

In [ ]:
# -- Head-to-head overlay: ALL 10 samples of each arm, shared scales, cropped -------
def render_h2h_overlay(frame):
    g = np.asarray(gt[frame, CH])
    P = {l: np.asarray(pred[l][frame, :, CH]) for l in H2H}
    K = max(p.shape[0] for p in P.values())
    err_max = float(np.percentile(np.stack([np.abs(P[l].mean(0) - g) for l in H2H]), 99)) or 1.0
    std_max = float(np.percentile(np.stack([P[l].std(0) for l in H2H]), 99)) or 1.0
    box = _melt_box(g)
    titles = (['ground truth'] + [f'sample {k + 1}' for k in range(K)]
              + ['ensemble mean', f'|mean - GT|\n0-{err_max:.0f} K (p99)', f'ensemble std\n0-{std_max:.0f} K (p99)'])
    fig, axes = plt.subplots(len(H2H), len(titles), figsize=(1.55 * len(titles), 1.9 * len(H2H) + 0.6),
                             squeeze=False)
    for r, l in enumerate(H2H):
        p, m = P[l], P[l].mean(0)
        imgs = ([(g, 'jet', VMIN_T, VMAX_T)] + [(p[k], 'jet', VMIN_T, VMAX_T) for k in range(K)]
                + [(m, 'jet', VMIN_T, VMAX_T), (np.abs(m - g), 'magma', 0, err_max), (p.std(0), 'hot', 0, std_max)])
        for c, (img, cmap, vmin, vmax) in enumerate(imgs):
            ax = axes[r, c]
            ax.imshow(img.T, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax)
            if box:
                ax.set_xlim(box[0], box[1]); ax.set_ylim(box[2], box[3])
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(titles[c], fontsize=7)
        axes[r, 0].set_ylabel(f"{H2H_NAME[l]}\n{l}", fontsize=7, color=H2H_COLOR[l])
    name = next((k for k, f in H2H_FRAMES.items() if f == frame), 'selected frame')
    fig.suptitle(f'Frame {frame} ({name}): every sample, fixed scale {VMIN_T}-{VMAX_T} K', fontsize=10)
    plt.tight_layout()
    plt.show()

show_frames(render_h2h_overlay)

In [ ]:
# -- Head-to-head melt-pool boundaries: ensemble means, then all 10 samples per arm --
def render_h2h_contours(frame):
    g = np.asarray(gt[frame, CH])
    box = _melt_box(g)
    fig, axes = plt.subplots(1, 1 + len(H2H), figsize=(4.6 * (1 + len(H2H)), 4.6))
    for ax in axes:
        ax.imshow(g.T, origin='lower', cmap='Greys_r', vmin=VMIN_T, vmax=VMAX_T)
        if g.max() > MP_THRESHOLD_K:
            ax.contour(g.T, levels=[MP_THRESHOLD_K], colors='k', linewidths=2.5)
        if box:
            ax.set_xlim(box[0], box[1]); ax.set_ylim(box[2], box[3])
    handles = [plt.Line2D([], [], color='k', lw=2.5, label='ground truth')]
    for l in H2H:
        m = np.asarray(pred[l][frame, :, CH]).mean(0)
        if m.max() > MP_THRESHOLD_K:
            axes[0].contour(m.T, levels=[MP_THRESHOLD_K], colors=[H2H_COLOR[l]], linewidths=1.6)
        handles.append(plt.Line2D([], [], color=H2H_COLOR[l], lw=1.6, label=f"{H2H_NAME[l]}: {l}"))
    axes[0].legend(handles=handles, fontsize=7, loc='lower left')
    axes[0].set_title('ensemble-mean boundary (1900 K)', fontsize=10)
    for ax, l in zip(axes[1:], H2H):
        p = np.asarray(pred[l][frame, :, CH])
        for k in range(len(p)):
            if p[k].max() > MP_THRESHOLD_K:
                ax.contour(p[k].T, levels=[MP_THRESHOLD_K], colors=[H2H_COLOR[l]], linewidths=0.8, alpha=0.75)
        ax.set_title(f"{H2H_NAME[l]}: all {len(p)} sample boundaries", fontsize=10, color=H2H_COLOR[l])
    name = next((k for k, f in H2H_FRAMES.items() if f == frame), 'selected frame')
    fig.suptitle(f'Frame {frame} ({name}): black = ground truth', fontsize=10)
    plt.tight_layout()
    plt.show()

show_frames(render_h2h_contours)

**Reading the head-to-head.** In the overlay, compare samples *within* a row: where they differ, that arm is
uncertain; the last column shows the same thing as a map. In the boundary panels, a tight bundle of lines is a
confident arm and a fanned-out bundle an uncertain one; an arm is honestly uncertain when the black ground-truth line
lies *inside* its bundle, and over-confident when a tight bundle misses it. Single frames are illustrations; the table
above (calibration in particular) is the test-set-wide evidence.